# Parte 3: Modelos de Machine Learning

**Disciplina**: Big Data  
**Referencia**: Baldi, P., Sadowski, P. & Whiteson, D. *Searching for exotic particles in high-energy physics with deep learning*. Nature Communications 5, 4308 (2014).

Este notebook treina tres modelos de classificacao sobre o dataset SUSY e compara o desempenho usando dois conjuntos de features:

- **8 low-level**: medicoes brutas do detector (momento, angulos, energia perdida)
- **18 todas**: as 8 acima mais as 10 variaveis derivadas manualmente por fisicos

O experimento replica a pergunta central do paper: uma rede neural consegue aprender as representacoes que fisicos derivaram manualmente?  
Benchmark do paper (deep learning): AUC 0.876 com 8 features, AUC 0.885 com 18 features.

## Setup

Carregamos as bibliotecas do Spark ML e iniciamos a sessao. O ponto de entrada e o Parquet gerado na Parte 2, que contem os dados ja limpos e balanceados sem precisar reler o CSV de 1.61 GB.

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (DecisionTreeClassifier,
                                        LogisticRegression,
                                        MultilayerPerceptronClassifier)
from pyspark.ml.evaluation import (MulticlassClassificationEvaluator,
                                    BinaryClassificationEvaluator)
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

spark = SparkSession.builder \
    .appName("SUSY-AC2-Modelos") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"Spark {spark.version} pronto.")

Spark 3.5.0 pronto.


In [2]:
df = spark.read.parquet("./data/susy_parquet")

print(f"Linhas carregadas: {df.count():,}")
df.printSchema()

Linhas carregadas: 4,575,763
root
 |-- label: integer (nullable = true)
 |-- lepton1_pT: double (nullable = true)
 |-- lepton1_eta: double (nullable = true)
 |-- lepton1_phi: double (nullable = true)
 |-- lepton2_pT: double (nullable = true)
 |-- lepton2_eta: double (nullable = true)
 |-- lepton2_phi: double (nullable = true)
 |-- missing_energy_magnitude: double (nullable = true)
 |-- missing_energy_phi: double (nullable = true)
 |-- MET_rel: double (nullable = true)
 |-- axial_MET: double (nullable = true)
 |-- M_R: double (nullable = true)
 |-- M_TR_2: double (nullable = true)
 |-- R: double (nullable = true)
 |-- MT2: double (nullable = true)
 |-- S_R: double (nullable = true)
 |-- M_Delta_R: double (nullable = true)
 |-- dPhi_r_b: double (nullable = true)
 |-- cos_theta_r1: double (nullable = true)



## 3.1. Definicao dos Conjuntos de Features

Os dois conjuntos replicam as configuracoes experimentais do paper. Cada modelo sera treinado duas vezes, uma para cada conjunto, permitindo comparar diretamente o impacto das features derivadas.

In [3]:
low_level_cols = [
    "lepton1_pT", "lepton1_eta", "lepton1_phi",
    "lepton2_pT", "lepton2_eta", "lepton2_phi",
    "missing_energy_magnitude", "missing_energy_phi"
]

all_cols = [c for c in df.columns if c != "label"]

feature_sets = {
    "8 low-level": low_level_cols,
    "18 todas"   : all_cols,
}

print(f"Conjunto low-level ({len(low_level_cols)} features): {low_level_cols}")
print(f"Conjunto completo  ({len(all_cols)} features): {all_cols}")

Conjunto low-level (8 features): ['lepton1_pT', 'lepton1_eta', 'lepton1_phi', 'lepton2_pT', 'lepton2_eta', 'lepton2_phi', 'missing_energy_magnitude', 'missing_energy_phi']
Conjunto completo  (18 features): ['lepton1_pT', 'lepton1_eta', 'lepton1_phi', 'lepton2_pT', 'lepton2_eta', 'lepton2_phi', 'missing_energy_magnitude', 'missing_energy_phi', 'MET_rel', 'axial_MET', 'M_R', 'M_TR_2', 'R', 'MT2', 'S_R', 'M_Delta_R', 'dPhi_r_b', 'cos_theta_r1']


## 3.2. Divisao Treino e Teste

Um unico split 80/20 e aplicado sobre os dados brutos do Parquet. Os dois experimentos (8 e 18 features) usam exatamente os mesmos conjuntos de treino e teste, garantindo comparacao justa.

O `seed=42` assegura reproducibilidade: qualquer reexecucao produz a mesma divisao.

In [4]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
train_df.cache()
test_df.cache()

train_count = train_df.count()
test_count  = test_df.count()
total = train_count + test_count
print(f"Treino : {train_count:,} ({train_count/total:.0%})")
print(f"Teste  : {test_count:,} ({test_count/total:.0%})")

Treino : 3,660,733 (80%)
Teste  : 915,030 (20%)


## 3.3. Pipeline de Treino e Avaliacao

A funcao `treinar_modelo` monta um `Pipeline` com tres estagios:

1. `VectorAssembler`: concatena as colunas selecionadas em um vetor `"features"`
2. `StandardScaler`: normaliza para media zero e desvio padrao 1
3. Modelo: recebe `"scaled_features"` como entrada

O `StandardScaler` e aplicado em todos os modelos, incluindo a Arvore de Decisao. A decisao e baseada na EDA da Parte 2: as features tem escalas muito diferentes (`lepton_pT` ate ~20, `phi` entre -pi e pi, `cos_theta_r1` entre 0 e 1). Normalizar nao prejudica a Arvore (que usa apenas comparacoes relativas para definir splits) e garante um Pipeline unico e consistente para os tres modelos.

In [5]:
def treinar_modelo(model, feature_cols, train_data=None):
    if train_data is None:
        train_data = train_df
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    scaler    = StandardScaler(inputCol="features", outputCol="scaled_features",
                               withMean=True, withStd=True)
    pipeline  = Pipeline(stages=[assembler, scaler, model])
    fitted    = pipeline.fit(train_data)
    preds     = fitted.transform(test_df)

    acc = MulticlassClassificationEvaluator(
              labelCol="label", metricName="accuracy").evaluate(preds)
    f1  = MulticlassClassificationEvaluator(
              labelCol="label", metricName="f1").evaluate(preds)
    auc = BinaryClassificationEvaluator(
              labelCol="label", metricName="areaUnderROC").evaluate(preds)

    return {"Accuracy": round(acc, 4), "F1": round(f1, 4), "AUC-ROC": round(auc, 4)}, preds

## 3.4. Arvore de Decisao

A Arvore de Decisao aprende uma sequencia de regras "se/entao" sobre os valores das features, particionando o espaco de dados recursivamente. E o modelo mais interpretavel dos tres: e possivel inspecionar as regras aprendidas.

Parametros:
- `maxDepth=10`: limita a profundidade da arvore para evitar overfitting
- `seed=42`: reproducibilidade das escolhas aleatorias internas
- Embora a Arvore de Decisao nao seja sensivelmente afetada pela escala das features, ela recebe `scaled_features` por consistencia com os outros modelos no Pipeline.

In [6]:
resultados_dt = {}
preds_dt = {}

dt = DecisionTreeClassifier(labelCol="label", featuresCol="scaled_features",
                             maxDepth=10, seed=42)

for nome_feat, feat_cols in feature_sets.items():
    print(f"Treinando Arvore de Decisao | {nome_feat}...")
    metricas, preds = treinar_modelo(dt, feat_cols)
    resultados_dt[nome_feat] = metricas
    preds_dt[nome_feat] = preds
    print(f"  Accuracy: {metricas['Accuracy']:.4f} | F1: {metricas['F1']:.4f} | AUC-ROC: {metricas['AUC-ROC']:.4f}")

Treinando Arvore de Decisao | 8 low-level...
  Accuracy: 0.7758 | F1: 0.7753 | AUC-ROC: 0.4275
Treinando Arvore de Decisao | 18 todas...
  Accuracy: 0.7841 | F1: 0.7832 | AUC-ROC: 0.4626


## 3.5. Regressao Logistica

A Regressao Logistica e um modelo linear: aprende um peso para cada feature e usa a funcao sigmoidepara converter a combinacao linear em probabilidade. E sensivelmente afetada pela escala das features, por isso o `StandardScaler` e especialmente importante aqui.

Parametros:
- `maxIter=100`: numero maximo de iteracoes do otimizador (LBFGS)
- `regParam=0.01`: regularizacao L2 para evitar overfitting

In [7]:
resultados_lr = {}
preds_lr = {}

lr = LogisticRegression(labelCol="label", featuresCol="scaled_features",
                        maxIter=100, regParam=0.01)

for nome_feat, feat_cols in feature_sets.items():
    print(f"Treinando Regressao Logistica | {nome_feat}...")
    metricas, preds = treinar_modelo(lr, feat_cols)
    resultados_lr[nome_feat] = metricas
    preds_lr[nome_feat] = preds
    print(f"  Accuracy: {metricas['Accuracy']:.4f} | F1: {metricas['F1']:.4f} | AUC-ROC: {metricas['AUC-ROC']:.4f}")

Treinando Regressao Logistica | 8 low-level...
  Accuracy: 0.7614 | F1: 0.7597 | AUC-ROC: 0.8321
Treinando Regressao Logistica | 18 todas...
  Accuracy: 0.7766 | F1: 0.7751 | AUC-ROC: 0.8517


### Estimativa de memória por arquitetura

Antes de escolher a arquitetura, calculamos o consumo teórico de memória.
Os três componentes principais são:
- **L-BFGS (driver)**: `(2 + 2m) × P × 8 bytes`, onde m=10 e P = total de parâmetros
- **Ativações por bloco (executor)**: `2 × blockSize × Σneurônios × 8 bytes`
- **Overhead do Spark**: ~40% da memória configurada reservada para GC e runtime

A memória disponível para computação é `spark.driver.memory × 0.6`.

In [8]:
def estimar_memoria_mlp(layers, block_size=256, lbfgs_m=10, driver_gb=4):
    params = sum(
        layers[i] * layers[i+1] + layers[i+1]
        for i in range(len(layers) - 1)
    )
    lbfgs_mb       = (2 + 2 * lbfgs_m) * params * 8 / 1024**2
    activations_mb = 2 * block_size * sum(layers) * 8 / 1024**2
    total_mb       = lbfgs_mb + activations_mb
    disponivel_mb  = driver_gb * 1024 * 0.6
    status = "OK" if total_mb < disponivel_mb else "EXCEDE MEMORIA"
    return dict(params=params, lbfgs_mb=round(lbfgs_mb, 1),
                activations_mb=round(activations_mb, 1),
                total_mb=round(total_mb, 1),
                disponivel_mb=round(disponivel_mb, 1), status=status)

arquiteturas = [
    [8,  300, 300, 2],
    [8,  200, 100, 2],
    [8,  100,  50, 2],
    [18, 300, 300, 2],
    [18, 200, 100, 2],
    [18, 100,  50, 2],
]

print(f"{'Arquitetura':<22} {'Params':>8} {'L-BFGS':>9} {'Ativ.':>8} {'Total':>8} {'Disponivel':>11} Status")
print("-" * 82)
for arch in arquiteturas:
    r = estimar_memoria_mlp(arch, block_size=256, driver_gb=4)
    print(f"{str(arch):<22} {r['params']:>8,} {r['lbfgs_mb']:>8.1f}MB"
          f" {r['activations_mb']:>7.1f}MB {r['total_mb']:>7.1f}MB"
          f" {r['disponivel_mb']:>10.0f}MB  {r['status']}")

Arquitetura              Params    L-BFGS    Ativ.    Total  Disponivel Status
----------------------------------------------------------------------------------
[8, 300, 300, 2]         93,602     15.7MB     2.4MB    18.1MB       2458MB  OK
[8, 200, 100, 2]         22,102      3.7MB     1.2MB     4.9MB       2458MB  OK
[8, 100, 50, 2]           6,052      1.0MB     0.6MB     1.6MB       2458MB  OK
[18, 300, 300, 2]        96,602     16.2MB     2.4MB    18.6MB       2458MB  OK
[18, 200, 100, 2]        24,102      4.0MB     1.2MB     5.3MB       2458MB  OK
[18, 100, 50, 2]          7,052      1.2MB     0.7MB     1.8MB       2458MB  OK


## 3.6. Rede Neural (MultilayerPerceptronClassifier)

O `MultilayerPerceptronClassifier` do PySpark usa **L-BFGS** (otimizador full-batch): cada iteracao percorre o dataset inteiro para calcular o gradiente. O custo por iteracao e proporcional ao tamanho dos dados **e** ao numero de parametros — independente de reducoes anteriores.

**Restricoes aplicadas para viabilizar o treinamento local:**

| Parametro | Antes | Agora | Efeito |
|---|---|---|---|
| Amostra de treino | 2 % (~73 k) | **0,5 % (~18 k)** | 4x menos dados por iteracao |
| Arquitetura | [n, 300, 300, 2] | **[n, 100, 50, 2]** | ~15x menos parametros |
| `maxIter` | 50 | **20** | 2,5x menos iteracoes |

Combinados, a reducao e de ~150x em relacao a configuracao original.

**Justificativa academica**: o paper Baldi et al. (2014) demonstra que redes neurais superam metodos classicos mesmo com menos dados — usar 0,5 % dos dados para ilustrar essa vantagem e coerente com a tese central do paper. O conjunto de **teste permanece completo** (915 k linhas) para avaliacao imparcial.

**Monitoramento**: thread auxiliar le o `SparkContext.statusTracker()` a cada 10 s e exibe a barra de progresso do stage ativo.

In [9]:
import time
import sys
import threading

# ── Amostra para MLP ──────────────────────────────────────────────────────────
SAMPLE_FRAC  = 0.005   # 0,5 % ≈ 18 k linhas
train_sample = train_df.sample(fraction=SAMPLE_FRAC, seed=42)
train_sample.cache()
n_sample = train_sample.count()
print(f"Amostra MLP: {n_sample:,} linhas "
      f"({SAMPLE_FRAC:.1%} dos {train_count:,} de treino completo)")


def _monitor_spark(sc, stop_ev, label, start_t):
    """Thread auxiliar: imprime progresso do stage ativo a cada 10 s."""
    last_report = [0.0]
    while not stop_ev.is_set():
        elapsed = time.time() - start_t
        if elapsed - last_report[0] >= 10:
            try:
                tracker = sc.statusTracker()
                sids    = tracker.getActiveStageIds()
                if sids:
                    info = tracker.getStageInfo(sids[0])
                    if info and info.numTasks() > 0:
                        done   = info.numCompletedTasks()
                        total  = info.numTasks()
                        pct    = done / total * 100
                        filled = int(pct / 5)
                        bar    = "█" * filled + "░" * (20 - filled)
                        print(f"  [{label}] {elapsed:5.0f}s  "
                              f"stage {sids[0]}: [{bar}] {pct:.0f}%  "
                              f"({done}/{total} tasks)")
                    else:
                        print(f"  [{label}] {elapsed:5.0f}s  "
                              f"stage {sids[0]}: iniciando...")
                else:
                    print(f"  [{label}] {elapsed:5.0f}s  entre iteracoes L-BFGS...")
                sys.stdout.flush()
            except Exception:
                pass
            last_report[0] = elapsed
        time.sleep(2)


# ── Treino ────────────────────────────────────────────────────────────────────
resultados_mlp = {}
preds_mlp      = {}
configs        = list(feature_sets.items())
tempos         = []

for i, (nome_feat, feat_cols) in enumerate(configs):
    layers = [len(feat_cols), 100, 50, 2]   # arquitetura compacta
    print(f"\n[{i+1}/{len(configs)}] MLP | {nome_feat} | "
          f"arquitetura {layers} | {n_sample:,} linhas ({SAMPLE_FRAC:.1%})")

    stop_ev = threading.Event()
    t0      = time.time()
    monitor = threading.Thread(
        target=_monitor_spark,
        args=(spark.sparkContext, stop_ev, nome_feat, t0),
        daemon=True
    )
    monitor.start()

    try:
        mlp = MultilayerPerceptronClassifier(
            labelCol="label", featuresCol="scaled_features",
            layers=layers, maxIter=20, blockSize=512, seed=42
        )
        metricas, preds = treinar_modelo(mlp, feat_cols, train_data=train_sample)
        elapsed = time.time() - t0
        tempos.append(elapsed)

        resultados_mlp[nome_feat] = metricas
        preds_mlp[nome_feat]      = preds

        restantes = len(configs) - (i + 1)
        eta = (sum(tempos) / len(tempos)) * restantes
        print(f"  [OK] {elapsed/60:.1f} min | "
              f"Accuracy {metricas['Accuracy']:.4f} | "
              f"F1 {metricas['F1']:.4f} | "
              f"AUC-ROC {metricas['AUC-ROC']:.4f}")
        if restantes:
            print(f"  ETA restante: ~{eta/60:.1f} min")

    except Exception as e:
        elapsed = time.time() - t0
        print(f"  [ERRO] {elapsed/60:.1f} min — {e}")
        resultados_mlp[nome_feat] = None
        preds_mlp[nome_feat]      = None

    finally:
        stop_ev.set()
        monitor.join(timeout=5)

Amostra MLP: 18,310 linhas (0.5% dos 3,660,733 de treino completo)

[1/2] MLP | 8 low-level | arquitetura [8, 100, 50, 2] | 18,310 linhas (0.5%)
  [OK] 0.1 min | Accuracy 0.7682 | F1 0.7682 | AUC-ROC 0.8433
  ETA restante: ~0.1 min

[2/2] MLP | 18 todas | arquitetura [18, 100, 50, 2] | 18,310 linhas (0.5%)
  [OK] 0.1 min | Accuracy 0.7766 | F1 0.7761 | AUC-ROC 0.8476


## 3.7. Avaliacao Comparativa

Consolidamos as metricas dos 6 treinos (3 modelos x 2 feature sets) em uma unica tabela. O verde destaca o melhor valor de AUC-ROC por conjunto de features.

Referencia do paper (deep learning com o dataset completo):
- AUC 0.876 com 8 features low-level
- AUC 0.885 com todas as 18 features

In [10]:
rows = []
for modelo, resultados in [("Arvore de Decisao",  resultados_dt),
                            ("Regressao Logistica", resultados_lr),
                            ("Rede Neural MLP",     resultados_mlp)]:
    for feat_nome, metricas in resultados.items():
        rows.append({"Modelo": modelo, "Features": feat_nome, **metricas})

comparison = pd.DataFrame(rows).set_index(["Modelo", "Features"])

benchmark = pd.DataFrame([
    {"Modelo": "Benchmark paper (deep learning)", "Features": "8 low-level",
     "Accuracy": "-", "F1": "-", "AUC-ROC": 0.876},
    {"Modelo": "Benchmark paper (deep learning)", "Features": "18 todas",
     "Accuracy": "-", "F1": "-", "AUC-ROC": 0.885},
]).set_index(["Modelo", "Features"])

print("=== Resultados completos ===")
display(pd.concat([comparison, benchmark]).style.highlight_max(
    subset=["AUC-ROC"], color="lightgreen"))

=== Resultados completos ===


In [11]:
modelos   = ["Arvore de Decisao", "Regressao Logistica", "Rede Neural MLP"]
auc_low   = [resultados_dt["8 low-level"]["AUC-ROC"],
             resultados_lr["8 low-level"]["AUC-ROC"],
             resultados_mlp["8 low-level"]["AUC-ROC"]]
auc_all   = [resultados_dt["18 todas"]["AUC-ROC"],
             resultados_lr["18 todas"]["AUC-ROC"],
             resultados_mlp["18 todas"]["AUC-ROC"]]

x   = np.arange(len(modelos))
w   = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, auc_low, w, label="8 low-level", color="steelblue")
b2 = ax.bar(x + w/2, auc_all, w, label="18 todas",    color="tomato")

ax.axhline(0.876, color="steelblue", linestyle="--", linewidth=1,
           label="Benchmark paper (8 features, AUC 0.876)")
ax.axhline(0.885, color="tomato",    linestyle="--", linewidth=1,
           label="Benchmark paper (18 features, AUC 0.885)")

ax.set_xticks(x)
ax.set_xticklabels(modelos)
ax.set_ylabel("AUC-ROC")
ax.set_title("AUC-ROC por Modelo e Conjunto de Features")
ax.set_ylim(0.5, 1.0)
ax.legend(fontsize=8)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("./docs/auc_comparativo.png", dpi=100, bbox_inches="tight")
plt.show()

## 3.8. Salvar Predicoes

Salvamos as predicoes de cada combinacao modelo x features em Parquet separado, para auditoria e eventuais analises de erro.

In [12]:
import os
os.makedirs("./output", exist_ok=True)

configs = [
    ("dt",  preds_dt),
    ("lr",  preds_lr),
    ("mlp", preds_mlp),
]

for nome_modelo, preds_dict in configs:
    for feat_nome, preds in preds_dict.items():
        sufixo  = "lowlevel" if feat_nome == "8 low-level" else "all"
        caminho = f"./output/predicoes_{nome_modelo}_{sufixo}"
        preds.select("label", "prediction", "probability") \
             .write.mode("overwrite").parquet(caminho)
        print(f"Salvo: {caminho}")

print("\nTodos os arquivos de predicao gerados.")

Salvo: ./output/predicoes_dt_lowlevel
Salvo: ./output/predicoes_dt_all
Salvo: ./output/predicoes_lr_lowlevel
Salvo: ./output/predicoes_lr_all
Salvo: ./output/predicoes_mlp_lowlevel
Salvo: ./output/predicoes_mlp_all

Todos os arquivos de predicao gerados.


## Resumo da Parte 3

| Modelo | Features | Accuracy | F1 | AUC-ROC | Benchmark paper |
|---|---|---|---|---|---|
| Arvore de Decisao | 8 low-level | (ver saida) | (ver saida) | (ver saida) | - |
| Arvore de Decisao | 18 todas | (ver saida) | (ver saida) | (ver saida) | - |
| Regressao Logistica | 8 low-level | (ver saida) | (ver saida) | (ver saida) | - |
| Regressao Logistica | 18 todas | (ver saida) | (ver saida) | (ver saida) | - |
| Rede Neural MLP | 8 low-level | (ver saida) | (ver saida) | (ver saida) | 0.876 |
| Rede Neural MLP | 18 todas | (ver saida) | (ver saida) | (ver saida) | 0.885 |

**Interpretacao esperada**: A diferenca de AUC entre 8 e 18 features deve ser pequena para o MLP (como demonstrado no paper) e potencialmente maior para DT e LR, pois esses modelos lineares/rascos se beneficiam mais das features derivadas que ja capturam a fisica relevante.